# Build a Changelog Generator from Git History — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build a Changelog Generator from Git History](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/changelog-from-git)
lesson, adapted to run in a hosted notebook: a small sample repo is created in place with
messy, realistic commit history, and a free-tier LLM turns it into a clean, categorized,
hash-cited changelog.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/changelog-from-git) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/changelog-from-git) for the real, file-based version of this same code.

## Step 0: Install dependencies

In [ ]:
!pip install openai

## Step 1: Create a sample git repo

A hosted notebook needs a real git repo to read. Instead of cloning, build a small one in
place with a handful of intentionally messy commits — features, fixes, docs, and noise
("wip", "tweak", a merge commit) — so the changelog tool has something realistic to chew on.

In [ ]:
import os
import subprocess
from pathlib import Path

os.makedirs("sample_repo", exist_ok=True)
os.chdir("sample_repo")

def git(*args):
    subprocess.run(["git", *args], check=True, capture_output=True)

git("init", "-q")
git("config", "user.email", "student@example.com")
git("config", "user.name", "Student")

STEPS = [
    ("feat: add user login with email verification", "app.py", "def login():\n    pass\n"),
    ("wip", "app.py", "def login():\n    return email\n"),
    ("fix: handle empty email in login", "app.py", "def login(email):\n    return email.strip() or None\n"),
    ("add README", "README.md", "# Sample App\n"),
    ("feat: password reset flow", "auth.py", "def reset_password():\n    pass\n"),
    ("Merge branch 'feature/reset'", "auth.py", "def reset_password():\n    return True\n"),
    ("fix: reset token expiry", "auth.py", "from datetime import timedelta\nTOKEN_TTL = timedelta(hours=1)\n"),
    ("refactor utils", "utils.py", "def normalize(s):\n    return s\n"),
    ("feat: dark mode toggle", "theme.py", "def toggle_theme():\n    pass\n"),
    ("tweak", "theme.py", "def toggle_theme():\n    return 'dark'\n"),
    ("feat: export CSV report", "reports.py", "def export_csv():\n    return []\n"),
    ("bump version to 1.2.0", "pyproject.toml", 'version = "1.2.0"\n'),
]

for subject, filename, content in STEPS:
    Path(filename).write_text(content)
    git("add", "-A")
    git("commit", "-q", "-m", subject)
print(f"Created sample repo with {len(STEPS)} commits")

## Step 2: Fetch the commit stream

Same as [`fetch_commits.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/changelog-from-git/fetch_commits.py):
`git log` with a NUL-separated `--format` so messy subjects can't break parsing.

In [ ]:
def _run_git(args: list[str]) -> str:
    result = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout


def load_commits(max_commits: int = 50) -> list[dict]:
    raw = _run_git([
        "log", f"-{max_commits}",
        "--format=%H%x00%an%x00%ad%x00%s%x00%x00",
        "--date=short",
    ])
    commits = []
    for block in raw.split("\x00\x00"):
        block = block.strip("\n")
        if not block:
            continue
        parts = block.split("\x00")
        if len(parts) < 4:
            continue
        commits.append({"hash": parts[0], "author": parts[1], "date": parts[2], "subject": parts[3]})
    return commits


for c in load_commits(50):
    print(f"  {c['hash'][:8]} {c['date']}  {c['author']}: {c['subject']}")

## Step 3: Get a free-tier LLM API key

Pick any provider from the table in the [lesson's Setup](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/changelog-from-git#get-a-free-llm-api-key) —
GitHub Models is the suggested default. Paste the key below; `getpass` keeps it out of the
notebook's saved output and cell history.

In [ ]:
import getpass

os.environ["GITHUB_TOKEN"] = getpass.getpass("Paste your GitHub Models API key: ")

## Step 4: Generate the changelog

Same prompt as [`generate.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/changelog-from-git/generate.py):
categorized sections, noise-filtering, and a **commit-hash citation per entry** so nothing
can be silently invented.

In [ ]:
from openai import OpenAI

PROMPT_TEMPLATE = """You are writing a release changelog from commit messages.
Below is a list of recent commits, each tagged with a short hash. Write a
clean changelog with these rules:

- Group entries into sections: **Added**, **Changed**, **Fixed**.
- Merge commits that clearly belong to the same change; drop pure noise
  (merge commits, "wip", formatting-only messages) unless they hint at a real
  change, in which case include the change.
- Every entry must end with the commit hash(es) it came from, like "(abc1234)".
- Do NOT invent commits, features, or fixes. If a message is too vague to
  classify, put it in a final "Other / unclear" section rather than guessing.

Commits (hash: subject):
{commits}

Changelog:
"""


def build_prompt(commits: list[dict]) -> str:
    lines = [f"{c['hash'][:8]}: {c['subject']}" for c in commits]
    return PROMPT_TEMPLATE.format(commits="\n".join(lines))


client = OpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.github.ai/inference",
)
response = client.chat.completions.create(
    model="gpt-4o-mini",  # confirm this still has a free tier before running
    messages=[{"role": "user", "content": build_prompt(load_commits(50))}],
)
print(response.choices[0].message.content)